In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Load dataset
df = pd.read_csv("final_ev_dataset_15000.csv")

In [3]:
# Clean missing values
df = df.dropna()

In [4]:
# Encode charging_type
le = LabelEncoder()
df["charging_type"] = le.fit_transform(df["charging_type"])

In [5]:
# Features and target
features = [
    "Power (kW)",
    "num_chargers",
    "voltage_level",
    "current_flow",
    "charging_type"
]

X = df[features]
y = df["power_consumed"]

In [6]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
# Models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []

best_model = None
best_score = -999

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })

    if r2 > best_score:
        best_score = r2
        best_model = model

In [8]:
# Feature importance for Random Forest
rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

In [9]:
# Save files
pickle.dump(best_model, open("ev_model.pkl", "wb"))
pickle.dump(le, open("charging_type_encoder.pkl", "wb"))
pickle.dump(features, open("features.pkl", "wb"))
pickle.dump(pd.DataFrame(results), open("model_results.pkl", "wb"))
pickle.dump(feature_importance, open("feature_importance.pkl", "wb"))

print("Model training completed successfully.")
print("Model expects features:", best_model.n_features_in_)
print(pd.DataFrame(results))

Model training completed successfully.
Model expects features: 5
               Model    MAE   RMSE  R2 Score
0  Linear Regression  17.64  23.48    0.9246
1      Decision Tree   1.54   2.12    0.9994
2      Random Forest   0.56   0.81    0.9999
